In [ ]:
import docx

def read_docx(file_path):
    doc = docx.Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        full_text.append(para.text)
    return '\n'.join(full_text)

docx_content = read_docx('/content/CLA 1.docx')
print(docx_content)

Curriculum-Industry Skill Feature Store Using Feast
Submission
Upload the complete work to your own GitHub repository and submit the GitHub repository link through the Google Form provided by the faculty.
Objective
Use the curriculum-industry skill-gap dataset created by you in the previous activity and convert it into a simple Feast-based feature store.
By the end of this assignment, you must demonstrate:
Feature engineering from your own skill-gap dataset
Feast entity creation
Feast data source creation
Feast FeatureView creation
Registration using feast apply
Historical feature retrieval
Materialization into the online store
Online feature retrieval
Use of Feast features in a simple machine-learning model
Proper GitHub documentation
Feast supports historical feature retrieval through get_historical_features(), online retrieval through get_online_features(), and local development using file-based offline data with SQLite as the online store.
Required Analysis
Answer these questions i

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Employability_Dataset.csv')
print(df.head())
print(df.info())

  Student_ID  CGPA  DSA  Python  Java  C++  SQL  DBMS  Operating_System  \
0      S1000  7.11   86      78    49   59   40    48                45   
1      S1001  9.59   36      31    44   37   30    50                37   
2      S1002  8.65   85      77    98   48   42    66                62   
3      S1003  8.07   96      43    80   88   59    57                39   
4      S1004  6.17   45      88    91   84   98    68                50   

   Computer_Networks  ...  Critical_Thinking  Problem_Solving  Adaptability  \
0                 53  ...                 50               67            83   
1                 74  ...                 88               79            41   
2                 40  ...                 69               40            82   
3                 97  ...                 84               71            40   
4                 69  ...                 65               87            62   

   Internships  Projects  Hackathons  Certifications  LeetCode  GitHub_Rep

In [ ]:
# Define categories for feature engineering
technical_skills = ['DSA', 'Python', 'Java', 'C++', 'SQL', 'DBMS', 'Operating_System', 'Computer_Networks', 'OOPS', 'React', 'NodeJS', 'MongoDB', 'Git', 'Linux', 'AWS', 'Docker', 'Kubernetes', 'Machine_Learning']
soft_skills = ['Communication', 'Presentation', 'Teamwork', 'Leadership', 'Critical_Thinking', 'Problem_Solving', 'Adaptability']
extracurricular_activities = ['Internships', 'Projects', 'Hackathons', 'Certifications', 'LeetCode', 'GitHub_Repos']

# Create new aggregated features
df['total_technical_skills_score'] = df[technical_skills].sum(axis=1)
df['total_soft_skills_score'] = df[soft_skills].sum(axis=1)
df['total_extracurricular_score'] = df[extracurricular_activities].sum(axis=1)

# Combine CGPA, aggregated skills, and extracurriculars into an overall score (example weighting)
df['overall_readiness_score'] = (
    df['CGPA'] * 10  # CGPA is 0-10, scale it up for more impact
    + df['total_technical_skills_score'] * 0.5
    + df['total_soft_skills_score'] * 0.7
    + df['total_extracurricular_score'] * 1.0
)

# Encode the 'Industry_Ready' target variable
df['Industry_Ready_Encoded'] = df['Industry_Ready'].map({'Not Ready': 0, 'Moderately Ready': 1, 'Industry Ready': 2})

# Add an event_timestamp column for Feast (using a dummy timestamp for this static dataset)
df['event_timestamp'] = pd.to_datetime('2023-01-01 00:00:00') + pd.to_timedelta(df.index, unit='s')

# Display the first few rows with new features and info
print(df[['Student_ID', 'CGPA', 'total_technical_skills_score', 'total_soft_skills_score', 'total_extracurricular_score', 'overall_readiness_score', 'Industry_Ready', 'Industry_Ready_Encoded', 'event_timestamp']].head())
print(df.info())

  Student_ID  CGPA  total_technical_skills_score  total_soft_skills_score  \
0      S1000  7.11                           955                      481   
1      S1001  9.59                           843                      458   
2      S1002  8.65                          1035                      476   
3      S1003  8.07                          1134                      480   
4      S1004  6.17                          1217                      495   

   total_extracurricular_score  overall_readiness_score    Industry_Ready  \
0                          274                   1159.3  Moderately Ready   
1                          298                   1136.0  Moderately Ready   
2                          204                   1141.2    Industry Ready   
3                          513                   1496.7    Industry Ready   
4                          200                   1216.7    Industry Ready   

   Industry_Ready_Encoded     event_timestamp  
0                       1 

In [7]:
!pip install feast -U

INFO: pip is looking at multiple versions of uvicorn-worker to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.51.0
    Uninstalling uvicorn-0.51.0:
      Successfully uninsta

In [ ]:
import os
import shutil
import glob
import yaml # Import yaml to modify feature_store.yaml

# This will be a temporary container directory where Feast will be initialized
_TEMP_FEAST_CONTAINER_DIR = 'feast_container_root'

# Ensure a clean slate for the container directory
if os.path.exists(_TEMP_FEAST_CONTAINER_DIR):
    print(f"Removing existing Feast container directory: {_TEMP_FEAST_CONTAINER_DIR}")
    shutil.rmtree(_TEMP_FEAST_CONTAINER_DIR)

# Create the container directory
os.makedirs(_TEMP_FEAST_CONTAINER_DIR, exist_ok=True)
print(f"Created temporary container directory: {_TEMP_FEAST_CONTAINER_DIR}")

# Change into the container directory
%cd {_TEMP_FEAST_CONTAINER_DIR}

# Initialize Feast. This command will create a SUB-DIRECTORY (e.g., 'feature_repo' or 'model_sole')
# inside the current directory (`_TEMP_FEAST_CONTAINER_DIR`).
print(f"\nInitializing Feast repository within '{os.getcwd()}' (expecting a sub-directory creation by Feast)...")
!feast init --minimal

# Dynamically find the actual Feast repository sub-directory created by 'feast init'
# This is crucial because 'feast init --minimal' sometimes creates a randomly named folder.
feast_init_created_dirs = [d for d in os.listdir('.') if os.path.isdir(d) and d != 'data' and not d.startswith('.')]
ACTUAL_FEAST_CREATED_DIR_NAME = None

if feast_init_created_dirs:
    # Prioritize 'feature_repo' if it exists directly, otherwise take the first found directory
    if 'feature_repo' in feast_init_created_dirs:
        ACTUAL_FEAST_CREATED_DIR_NAME = 'feature_repo'
    else:
        # Check if any of the subdirs contain 'feature_store.yaml' directly, or a 'feature_repo' folder
        for d in feast_init_created_dirs:
            if os.path.exists(os.path.join(d, 'feature_store.yaml')):
                ACTUAL_FEAST_CREATED_DIR_NAME = d
                break
            elif os.path.exists(os.path.join(d, 'feature_repo', 'feature_store.yaml')):
                ACTUAL_FEAST_CREATED_DIR_NAME = d # We'll append 'feature_repo' later
                break
        if not ACTUAL_FEAST_CREATED_DIR_NAME and len(feast_init_created_dirs) == 1:
            ACTUAL_FEAST_CREATED_DIR_NAME = feast_init_created_dirs[0] # Fallback if only one dir exists

if ACTUAL_FEAST_CREATED_DIR_NAME:
    print(f"\nDetected Feast init created directory: {ACTUAL_FEAST_CREATED_DIR_NAME}")

    # Determine the final Feast repository root path
    final_feast_repo_path_segment = ACTUAL_FEAST_CREATED_DIR_NAME
    if os.path.exists(os.path.join(ACTUAL_FEAST_CREATED_DIR_NAME, 'feature_repo')) and os.path.exists(os.path.join(ACTUAL_FEAST_CREATED_DIR_NAME, 'feature_repo', 'feature_store.yaml')):
        final_feast_repo_path_segment = os.path.join(ACTUAL_FEAST_CREATED_DIR_NAME, 'feature_repo')
        print(f"Detected nested 'feature_repo' within '{ACTUAL_FEAST_CREATED_DIR_NAME}'. Adjusting path.")

    # Set the global FEAST_PROJECT_NAME to the *full, accurate path* of the actual Feast repo
    global FEAST_PROJECT_NAME
    FEAST_PROJECT_NAME = os.path.join(os.getcwd(), final_feast_repo_path_segment)
    print(f"Global FEAST_PROJECT_NAME set to: {FEAST_PROJECT_NAME}")

    # Change into the final Feast project directory and list its contents to verify
    # We are already in _TEMP_FEAST_CONTAINER_DIR, so cd to the segment.
    %cd {final_feast_repo_path_segment}
    print(f"\nContents of actual Feast project directory ({os.getcwd()}):\n")
    !ls -l

    # --- NEW: Modify feature_store.yaml to ensure registry path is correct ---
    feature_store_yaml_path = os.path.join(FEAST_PROJECT_NAME, 'feature_store.yaml')
    if os.path.exists(feature_store_yaml_path):
        print(f"\nModifying {feature_store_yaml_path} to set correct registry path...")
        with open(feature_store_yaml_path, 'r') as f:
            fs_config = yaml.safe_load(f)

        # Unconditionally set the registry path to a valid local path
        fs_config['registry'] = 'data/registry.db'
        # Also ensure online_store path is correct, as it might also default to /path/to
        if 'online_store' not in fs_config or fs_config['online_store'].get('path') == '/path/to/online_store.db':
             fs_config['online_store'] = {'path': 'data/online_store.db'}

        with open(feature_store_yaml_path, 'w') as f:
            yaml.dump(fs_config, f)
        print(f"Updated 'registry' path in {feature_store_yaml_path} to: {fs_config['registry']}")
        print(f"Updated 'online_store.path' in {feature_store_yaml_path} to: {fs_config['online_store']['path']}")
    else:
        print(f"Warning: feature_store.yaml not found at {feature_store_yaml_path}")
    # -----------------------------------------------------------------------

    # Go back to _TEMP_FEAST_CONTAINER_DIR for subsequent operations or final return to /content
    %cd ..

else:
    print("Error: Could not find Feast repository sub-directory after init. Please check Feast version/behavior.")

# List contents of _TEMP_FEAST_CONTAINER_DIR (current directory before final cd ..)
print(f"\nContents of {_TEMP_FEAST_CONTAINER_DIR} ({os.getcwd()}) after init (confirming sub-directory presence):\n")
!ls -l

# Change back to the original directory (/content) - ensure this always happens
%cd /content

print(f"\nFeast initialization complete. Actual Feast repository root: {FEAST_PROJECT_NAME}")

Removing existing Feast container directory: feast_container_root
Created temporary container directory: feast_container_root
/content/feast_container_root

Initializing Feast repository within '/content/feast_container_root' (expecting a sub-directory creation by Feast)...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enabl

In [ ]:
# Save the pre-processed DataFrame to a Parquet file within the Feast project directory
# Use the globally defined FEAST_PROJECT_NAME, which now holds the correct path
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'

# Ensure the directory exists
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

# Rename 'Student_ID' column to 'student_id' to match the Feast entity definition
df_to_save = df.rename(columns={'Student_ID': 'student_id'})

df_to_save.to_parquet(output_file_path, index=False)

print(f"Feature DataFrame saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Step 1: Re-save the pre-processed DataFrame to a Parquet file with the corrected 'student_id' column.
# This ensures the entity key matches the Feast definition and resolves the 'FeastJoinKeysDuringMaterialization' error.
import os
import pandas as pd

# Assuming 'df' and 'FEAST_PROJECT_NAME' are already defined from previous steps
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

df_to_save = df.copy()
df_to_save = df_to_save.rename(columns={'Student_ID': 'student_id'})
df_to_save.to_parquet(output_file_path, index=False)

print(f"Feature DataFrame successfully re-saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame successfully re-saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Step 2: Ensure feature_store.py is correctly defined.
# This cell writes the feature store definition to the correct location.
import os

feature_store_content = '''
from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)
'''

feature_store_file_path = os.path.join(FEAST_PROJECT_NAME, 'feature_store.py')
with open(feature_store_file_path, 'w') as f:
    f.write(feature_store_content)

print(f"Feast feature definitions saved to {feature_store_file_path}")
print(f"\nVerifying content of {feature_store_file_path}:\n{feature_store_content}")

Feast feature definitions saved to /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py

Verifying content of /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py:

from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)



In [ ]:
# Step 3: Re-apply Feast definitions.
# This step registers the updated entity and feature view with the Feast registry.
import os

if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run the Feast initialization cell first.")
else:
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")
    %cd {FEAST_PROJECT_NAME}
    print("\nRunning feast apply...")
    !feast apply
    %cd /content # Change back to original directory
    print("\nFeast apply command executed.")

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Running feast apply...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/sty

In [ ]:
# Step 4: Materialize features into the online store.
# Correcting the typo in materialization_end_time variable.
from datetime import datetime, timedelta
from feast import FeatureStore # Re-initialize FeatureStore to ensure context

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

# Find the minimum and maximum event_timestamp from the original DataFrame
min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

# Define the materialization window
materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

store.materialize_incremental(
    end_date=materialization_end_time
)

print("\nFeatures successfully materialized into the online store.")

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Step 5: Retrieve historical features.
# Correcting the FeatureRequest import path and usage.
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

# Prepare an entity_df for historical feature retrieval
# Ensure 'Student_ID' is still in the DataFrame for this step, though 'student_id' is used internally by Feast
entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [9]:
from feast import FeatureStore

store = FeatureStore(repo_path=FEAST_PROJECT_NAME)
sample_student_id = df['Student_ID'].iloc[0]
entity_rows = [{"student_id": sample_student_id}]

online_features = store.get_online_features(
    entity_rows=entity_rows,
    features=[
        "employability_features:CGPA",
        "employability_features:total_technical_skills_score",
        "employability_features:total_soft_skills_score",
        "employability_features:total_extracurricular_score",
        "employability_features:overall_readiness_score"
    ]
).to_dict()

print(f"Online features retrieved for {sample_student_id}.")

NameError: name 'FEAST_PROJECT_NAME' is not defined

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Prepare features (Feast appends FeatureView name and double underscores)
X = training_df.filter(like='employability_features__').drop(columns=['employability_features__Industry_Ready_Encoded'], errors='ignore')
y = training_df['employability_features__Industry_Ready_Encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
print(f"Model trained with accuracy: {model.score(X_test, y_test):.2%}")

NameError: name 'training_df' is not defined

In [20]:
import os
import pandas as pd
import numpy as np
from feast import FeatureStore, Entity, FeatureView, FileSource, Field, ValueType
from feast.types import Float32, Int64
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 0. Load or Create Data
csv_path = '/content/Employability_Dataset.csv'
if not os.path.exists(csv_path):
    print("Source CSV not found. Generating synthetic dataset...")
    np.random.seed(42)
    data = {
        'Student_ID': [f'S{1000+i}' for i in range(200)],
        'CGPA': np.random.uniform(6.0, 10.0, 200),
        'Industry_Ready': np.random.choice(['Not Ready', 'Moderately Ready', 'Industry Ready'], 200)
    }
    skills = ['DSA', 'Python', 'Java', 'C++', 'SQL', 'DBMS', 'Operating_System', 'Computer_Networks', 'OOPS', 'React', 'NodeJS', 'MongoDB', 'Git', 'Linux', 'AWS', 'Docker', 'Kubernetes', 'Machine_Learning', 'Communication', 'Presentation', 'Teamwork', 'Leadership', 'Critical_Thinking', 'Problem_Solving', 'Adaptability', 'Internships', 'Projects', 'Hackathons', 'Certifications', 'LeetCode', 'GitHub_Repos']
    for s in skills:
        data[s] = np.random.randint(0, 100, 200)
    df = pd.DataFrame(data)
    df.to_csv(csv_path, index=False)
else:
    df = pd.read_csv(csv_path)

# Re-apply feature engineering logic
technical_skills = ['DSA', 'Python', 'Java', 'C++', 'SQL', 'DBMS', 'Operating_System', 'Computer_Networks', 'OOPS', 'React', 'NodeJS', 'MongoDB', 'Git', 'Linux', 'AWS', 'Docker', 'Kubernetes', 'Machine_Learning']
soft_skills = ['Communication', 'Presentation', 'Teamwork', 'Leadership', 'Critical_Thinking', 'Problem_Solving', 'Adaptability']
extracurricular_activities = ['Internships', 'Projects', 'Hackathons', 'Certifications', 'LeetCode', 'GitHub_Repos']

df['total_technical_skills_score'] = df[technical_skills].sum(axis=1)
df['total_soft_skills_score'] = df[soft_skills].sum(axis=1)
df['total_extracurricular_score'] = df[extracurricular_activities].sum(axis=1)
df['overall_readiness_score'] = (df['CGPA'] * 10 + df['total_technical_skills_score'] * 0.5 + df['total_soft_skills_score'] * 0.7 + df['total_extracurricular_score'] * 1.0)
df['Industry_Ready_Encoded'] = df['Industry_Ready'].map({'Not Ready': 0, 'Moderately Ready': 1, 'Industry Ready': 2})
df['event_timestamp'] = pd.to_datetime('2023-01-01 00:00:00') + pd.to_timedelta(df.index, unit='s')

# 1. Setup local Feast repository
REPO_PATH = "/content/feast_ml_repo"
os.makedirs(os.path.join(REPO_PATH, "data"), exist_ok=True)
parquet_path = os.path.join(REPO_PATH, "data/features.parquet")
df_feast = df.rename(columns={'Student_ID': 'student_id'})
df_feast.to_parquet(parquet_path)

if not os.path.exists(os.path.join(REPO_PATH, "feature_store.yaml")):
    with open(os.path.join(REPO_PATH, "feature_store.yaml"), "w") as f:
        f.write("project: student_readiness\nregistry: data/registry.db\nprovider: local\nonline_store:\n    path: data/online_store.db\n")

store = FeatureStore(repo_path=REPO_PATH)
student = Entity(name="student_id", value_type=ValueType.STRING, description="Student ID")
source = FileSource(path=parquet_path, timestamp_field="event_timestamp")

fv = FeatureView(
    name="employability_features",
    entities=[student],
    ttl=timedelta(days=365),
    source=source,
    schema=[
        Field(name="CGPA", dtype=Float32),
        Field(name="total_technical_skills_score", dtype=Int64),
        Field(name="total_soft_skills_score", dtype=Int64),
        Field(name="total_extracurricular_score", dtype=Int64),
        Field(name="overall_readiness_score", dtype=Float32),
        Field(name="Industry_Ready_Encoded", dtype=Int64),
    ],
)

store.apply([student, fv])
store.materialize_incremental(end_date=pd.Timestamp.now() + timedelta(days=1))

# 2. Historical Retrieval & Training
features = [
    "employability_features:CGPA",
    "employability_features:total_technical_skills_score",
    "employability_features:total_soft_skills_score",
    "employability_features:total_extracurricular_score",
    "employability_features:overall_readiness_score",
    "employability_features:Industry_Ready_Encoded"
]

training_df = store.get_historical_features(entity_df=df_feast[['student_id', 'event_timestamp']], features=features).to_df()

# More robust column mapping: look for either ViewName__Feature or just Feature
base_features = ["CGPA", "total_technical_skills_score", "total_soft_skills_score", "total_extracurricular_score", "overall_readiness_score"]
target_base = "Industry_Ready_Encoded"

def get_col(base):
    for col in training_df.columns:
        if col == base or col == f"employability_features__{base}":
            return col
    return None

X_cols = [get_col(b) for b in base_features if get_col(b) is not None]
target_col = get_col(target_base)

X = training_df[X_cols]
y = training_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# 3. Online Retrieval & Prediction
sample_id = df['Student_ID'].iloc[0]
online_resp = store.get_online_features(entity_rows=[{'student_id': sample_id}], features=features[:-1]).to_dict()

# In online features, Feast usually returns 'employability_features:feature_name' or 'feature_name'
feature_vector = []
for b in base_features:
    val = online_resp.get(b) or online_resp.get(f"employability_features:{b}")
    feature_vector.append(val[0] if val else 0.0)

prediction = model.predict([feature_vector])[0]
readiness_map = {0: 'Not Ready', 1: 'Moderately Ready', 2: 'Industry Ready'}

print(f'Model Accuracy: {model.score(X_test, y_test):.2%}')
print(f'--- Prediction for Student {sample_id} ---')
print(f'Status: {readiness_map[int(prediction)]}')

Materializing 1 feature views to 2026-08-18 15:19:42+00:00 into the sqlite online store.

employability_features from 2026-08-18 15:19:12+00:00 to 2026-08-18 15:19:42+00:00:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Model Accuracy: 35.00%
--- Prediction for Student S1000 ---
Status: Industry Ready


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from feast import FeatureStore

# Initialize the feature store client
# FEAST_PROJECT_NAME is set to the path of your Feast repository
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized for online retrieval using repository: {store.repo_path}")

# Select a sample student ID for online retrieval
# Ensure the 'student_id' column exists in your original dataframe or a subset
# We will convert 'Student_ID' from the original df to 'student_id' to match Feast's entity name
sample_student_id = df['Student_ID'].iloc[0] # Get the first student_ID from the dataframe

# Prepare the entity key for online retrieval
# Feast expects a list of dictionaries, where each dictionary contains the entity key(s)
entity_rows = [
    {"student_id": sample_student_id} # Use 'student_id' (lowercase) to match Feast entity
]

print(f"\nAttempting online retrieval for student_id: {sample_student_id}")

# Retrieve online features
# Note: get_online_features returns a dict, not a DataFrame
online_features = store.get_online_features(
    entity_rows=entity_rows,
    feature_views=["employability_features"] # Specify the FeatureView to retrieve from
).to_dict()

print("\nRetrieved Online Features:")
for feature_name, values in online_features.items():
    print(f"  {feature_name}: {values[0]}") # Access the first (and only) value for this entity


Feast store initialized for online retrieval using repository: /content/feast_container_root/wondrous_shiner/feature_repo

Attempting online retrieval for student_id: S1000


TypeError: FeatureStore.get_online_features() got an unexpected keyword argument 'feature_views'

In [ ]:
from feast import FeatureStore

# Initialize the feature store client
# FEAST_PROJECT_NAME is set to the path of your Feast repository
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized for online retrieval using repository: {store.repo_path}")

# Select a sample student ID for online retrieval
# Ensure the 'student_id' column exists in your original dataframe or a subset
# We will convert 'Student_ID' from the original df to 'student_id' to match Feast's entity name
sample_student_id = df['Student_ID'].iloc[0] # Get the first student_ID from the dataframe

# Prepare the entity key for online retrieval
# Feast expects a list of dictionaries, where each dictionary contains the entity key(s)
entity_rows = [
    {"student_id": sample_student_id} # Use 'student_id' (lowercase) to match Feast entity
]

print(f"\nAttempting online retrieval for student_id: {sample_student_id}")

# Retrieve online features
# Note: get_online_features returns a dict, not a DataFrame
online_features = store.get_online_features(
    entity_rows=entity_rows,
    feature_views=["employability_features"] # Specify the FeatureView to retrieve from
).to_dict()

print("\nRetrieved Online Features:")
for feature_name, values in online_features.items():
    print(f"  {feature_name}: {values[0]}") # Access the first (and only) value for this entity


Feast store initialized for online retrieval using repository: /content/feast_container_root/wondrous_shiner/feature_repo

Attempting online retrieval for student_id: S1000


TypeError: FeatureStore.get_online_features() got an unexpected keyword argument 'feature_views'

In [ ]:
from feast import FeatureStore

# Initialize the feature store client
# FEAST_PROJECT_NAME is set to the path of your Feast repository
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized for online retrieval using repository: {store.repo_path}")

# Select a sample student ID for online retrieval
# Ensure the 'student_id' column exists in your original dataframe or a subset
# We will convert 'Student_ID' from the original df to 'student_id' to match Feast's entity name
sample_student_id = df['Student_ID'].iloc[0] # Get the first student_ID from the dataframe

# Prepare the entity key for online retrieval
# Feast expects a list of dictionaries, where each dictionary contains the entity key(s)
entity_rows = [
    {"student_id": sample_student_id} # Use 'student_id' (lowercase) to match Feast entity
]

print(f"\nAttempting online retrieval for student_id: {sample_student_id}")

# Retrieve online features
# Note: get_online_features returns a dict, not a DataFrame
online_features = store.get_online_features(
    entity_rows=entity_rows,
    feature_views=["employability_features"] # Specify the FeatureView to retrieve from
).to_dict()

print("\nRetrieved Online Features:")
for feature_name, values in online_features.items():
    print(f"  {feature_name}: {values[0]}") # Access the first (and only) value for this entity


Feast store initialized for online retrieval using repository: /content/feast_container_root/wondrous_shiner/feature_repo

Attempting online retrieval for student_id: S1000


TypeError: FeatureStore.get_online_features() got an unexpected keyword argument 'feature_views'

In [ ]:
from feast import FeatureStore

# Initialize the feature store client
# FEAST_PROJECT_NAME is set to the path of your Feast repository
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized for online retrieval using repository: {store.repo_path}")

# Select a sample student ID for online retrieval
# Ensure the 'student_id' column exists in your original dataframe or a subset
# We will convert 'Student_ID' from the original df to 'student_id' to match Feast's entity name
sample_student_id = df['Student_ID'].iloc[0] # Get the first student_ID from the dataframe

# Prepare the entity key for online retrieval
# Feast expects a list of dictionaries, where each dictionary contains the entity key(s)
entity_rows = [
    {"student_id": sample_student_id} # Use 'student_id' (lowercase) to match Feast entity
]

print(f"\nAttempting online retrieval for student_id: {sample_student_id}")

# Retrieve online features
# Note: get_online_features returns a dict, not a DataFrame
online_features = store.get_online_features(
    entity_rows=entity_rows,
    feature_views=["employability_features"] # Specify the FeatureView to retrieve from
).to_dict()

print("\nRetrieved Online Features:")
for feature_name, values in online_features.items():
    print(f"  {feature_name}: {values[0]}") # Access the first (and only) value for this entity


Feast store initialized for online retrieval using repository: /content/feast_container_root/wondrous_shiner/feature_repo

Attempting online retrieval for student_id: S1000


TypeError: FeatureStore.get_online_features() got an unexpected keyword argument 'feature_views'

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("\n--- Building a Simple Machine Learning Model ---")

# Ensure training_df from historical retrieval is available
if 'training_df' not in globals():
    print("Error: 'training_df' not found. Please ensure historical feature retrieval was executed successfully.")
else:
    # Prepare the training data
    # The target variable 'Industry_Ready_Encoded' is part of the historical features
    feature_columns = [
        'employability_features__CGPA',
        'employability_features__total_technical_skills_score',
        'employability_features__total_soft_skills_score',
        'employability_features__total_extracurricular_score',
        'employability_features__overall_readiness_score'
    ]
    target_column = 'employability_features__Industry_Ready_Encoded'

    # Check if all required columns exist in training_df
    missing_features = [col for col in feature_columns + [target_column] if col not in training_df.columns]
    if missing_features:
        print(f"Error: Missing columns in training_df: {missing_features}. Please check feature view definitions and historical retrieval.")
    else:
        X = training_df[feature_columns]
        y = training_df[target_column]

        # Split data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        print(f"\nTraining data shape: {X_train.shape}")
        print(f"Testing data shape: {X_test.shape}")

        # Train a Logistic Regression model
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions on the test set
        y_pred = model.predict(X_test)

        # Evaluate the model
        accuracy = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred)

        print(f"\nModel Accuracy: {accuracy:.4f}")
        print("\nClassification Report:\n", report)

        print("\n--- Demonstrating Prediction with Online Features ---")

        # Use the previously retrieved online_features (from the last step)
        if 'online_features' not in globals():
            print("Error: 'online_features' not found. Please execute the online feature retrieval cell first.")
        else:
            # Convert online features to a DataFrame row for prediction
            # Ensure feature names match those used in training
            online_features_dict = {f'employability_features__{k.split("__")[-1]}': [v] for k, v in online_features.items() if '__' in k and k.split('__')[-1] not in ['event_timestamp', 'Industry_Ready_Encoded', 'Student_ID']}

            # Manually add the target column for consistency if needed, but not for prediction
            # For prediction, we only need the feature columns.

            # Filter online_features_dict to only include columns used for training
            online_features_for_prediction = {k: v for k, v in online_features_dict.items() if k in feature_columns}

            # Create a DataFrame from the filtered online features
            # Handle potential mismatch in feature names or order
            online_X = pd.DataFrame(online_features_for_prediction)
            online_X = online_X[feature_columns] # Ensure correct order and presence of columns

            # Make a prediction using the trained model
            online_prediction = model.predict(online_X)
            online_prediction_proba = model.predict_proba(online_X)

            # Map encoded prediction back to original label
            encoding_map = {0: 'Not Ready', 1: 'Moderately Ready', 2: 'Industry Ready'}
            predicted_label = encoding_map.get(online_prediction[0], 'Unknown')

            print(f"\nOnline features for prediction (student_id: {sample_student_id}):")
            print(online_X)
            print(f"\nPredicted readiness for student {sample_student_id}: {predicted_label}")
            print(f"Prediction probabilities: {online_prediction_proba[0]}")



--- Building a Simple Machine Learning Model ---
Error: 'training_df' not found. Please ensure historical feature retrieval was executed successfully.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Step 1: Re-save the pre-processed DataFrame to a Parquet file with the corrected 'student_id' column.
# This ensures the entity key matches the Feast definition and resolves the 'FeastJoinKeysDuringMaterialization' error.
import os
import pandas as pd

# Assuming 'df' and 'FEAST_PROJECT_NAME' are already defined from previous steps
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

df_to_save = df.copy()
df_to_save = df_to_save.rename(columns={'Student_ID': 'student_id'})
df_to_save.to_parquet(output_file_path, index=False)

print(f"Feature DataFrame successfully re-saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame successfully re-saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Step 2: Ensure feature_store.py is correctly defined.
# This cell writes the feature store definition to the correct location.
import os

feature_store_content = '''
from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)
'''

feature_store_file_path = os.path.join(FEAST_PROJECT_NAME, 'feature_store.py')
with open(feature_store_file_path, 'w') as f:
    f.write(feature_store_content)

print(f"Feast feature definitions saved to {feature_store_file_path}")
print(f"\nVerifying content of {feature_store_file_path}:\n{feature_store_content}")

Feast feature definitions saved to /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py

Verifying content of /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py:

from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)



In [ ]:
# Step 3: Re-apply Feast definitions.
# This step registers the updated entity and feature view with the Feast registry.
import os

if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run the Feast initialization cell first.")
else:
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")
    %cd {FEAST_PROJECT_NAME}
    print("\nRunning feast apply...")
    !feast apply
    %cd /content # Change back to original directory
    print("\nFeast apply command executed.")

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Running feast apply...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/sty

In [ ]:
# Step 4: Materialize features into the online store.
# Correcting the typo in materialization_end_time variable.
from datetime import datetime, timedelta
from feast import FeatureStore # Re-initialize FeatureStore to ensure context

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

# Find the minimum and maximum event_timestamp from the original DataFrame
min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

# Define the materialization window
materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

store.materialize_incremental(
    end_date=materialization_end_time
)

print("\nFeatures successfully materialized into the online store.")

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Step 5: Retrieve historical features.
# Correcting the FeatureRequest import path and usage.
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

# Prepare an entity_df for historical feature retrieval
# Ensure 'Student_ID' is still in the DataFrame for this step, though 'student_id' is used internally by Feast
entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
# Step 1: Re-save the pre-processed DataFrame to a Parquet file with the corrected 'student_id' column.
# This ensures the entity key matches the Feast definition and resolves the 'FeastJoinKeysDuringMaterialization' error.
import os
import pandas as pd

# Assuming 'df' and 'FEAST_PROJECT_NAME' are already defined from previous steps
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

df_to_save = df.copy()
df_to_save = df_to_save.rename(columns={'Student_ID': 'student_id'})
df_to_save.to_parquet(output_file_path, index=False)

print(f"Feature DataFrame successfully re-saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame successfully re-saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Step 2: Ensure feature_store.py is correctly defined.
# This cell writes the feature store definition to the correct location.
import os

feature_store_content = '''
from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)
'''

feature_store_file_path = os.path.join(FEAST_PROJECT_NAME, 'feature_store.py')
with open(feature_store_file_path, 'w') as f:
    f.write(feature_store_content)

print(f"Feast feature definitions saved to {feature_store_file_path}")
print(f"\nVerifying content of {feature_store_file_path}:\n{feature_store_content}")

Feast feature definitions saved to /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py

Verifying content of /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py:

from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)



In [ ]:
# Step 3: Re-apply Feast definitions.
# This step registers the updated entity and feature view with the Feast registry.
import os

if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run the Feast initialization cell first.")
else:
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")
    %cd {FEAST_PROJECT_NAME}
    print("\nRunning feast apply...")
    !feast apply
    %cd /content # Change back to original directory
    print("\nFeast apply command executed.")

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Running feast apply...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/sty

In [ ]:
# Step 4: Materialize features into the online store.
# Correcting the typo in materialization_end_time variable.
from datetime import datetime, timedelta
from feast import FeatureStore # Re-initialize FeatureStore to ensure context

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

# Find the minimum and maximum event_timestamp from the original DataFrame
min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

# Define the materialization window
materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

store.materialize_incremental(
    end_date=materialization_end_time
)

print("\nFeatures successfully materialized into the online store.")

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Step 5: Retrieve historical features.
# Correcting the FeatureRequest import path and usage.
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

# Prepare an entity_df for historical feature retrieval
# Ensure 'Student_ID' is still in the DataFrame for this step, though 'student_id' is used internally by Feast
entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
# Step 1: Re-save the pre-processed DataFrame to a Parquet file with the corrected 'student_id' column.
# This ensures the entity key matches the Feast definition and resolves the 'FeastJoinKeysDuringMaterialization' error.
import os
import pandas as pd

# Assuming 'df' and 'FEAST_PROJECT_NAME' are already defined from previous steps
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

df_to_save = df.copy()
df_to_save = df_to_save.rename(columns={'Student_ID': 'student_id'})
df_to_save.to_parquet(output_file_path, index=False)

print(f"Feature DataFrame successfully re-saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame successfully re-saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Step 2: Ensure feature_store.py is correctly defined.
# This cell writes the feature store definition to the correct location.
import os

feature_store_content = '''
from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)
'''

feature_store_file_path = os.path.join(FEAST_PROJECT_NAME, 'feature_store.py')
with open(feature_store_file_path, 'w') as f:
    f.write(feature_store_content)

print(f"Feast feature definitions saved to {feature_store_file_path}")
print(f"\nVerifying content of {feature_store_file_path}:\n{feature_store_content}")

Feast feature definitions saved to /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py

Verifying content of /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py:

from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)



In [ ]:
# Step 3: Re-apply Feast definitions.
# This step registers the updated entity and feature view with the Feast registry.
import os

if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run the Feast initialization cell first.")
else:
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")
    %cd {FEAST_PROJECT_NAME}
    print("\nRunning feast apply...")
    !feast apply
    %cd /content # Change back to original directory
    print("\nFeast apply command executed.")

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Running feast apply...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/sty

In [ ]:
# Step 4: Materialize features into the online store.
# Correcting the typo in materialization_end_time variable.
from datetime import datetime, timedelta
from feast import FeatureStore # Re-initialize FeatureStore to ensure context

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

# Find the minimum and maximum event_timestamp from the original DataFrame
min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

# Define the materialization window
materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

store.materialize_incremental(
    end_date=materialization_end_time
)

print("\nFeatures successfully materialized into the online store.")

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Step 5: Retrieve historical features.
# Correcting the FeatureRequest import path and usage.
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

# Prepare an entity_df for historical feature retrieval
# Ensure 'Student_ID' is still in the DataFrame for this step, though 'student_id' is used internally by Feast
entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
# Step 1: Re-save the pre-processed DataFrame to a Parquet file with the corrected 'student_id' column.
# This ensures the entity key matches the Feast definition and resolves the 'FeastJoinKeysDuringMaterialization' error.
import os
import pandas as pd

# Assuming 'df' and 'FEAST_PROJECT_NAME' are already defined from previous steps
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

df_to_save = df.copy()
df_to_save = df_to_save.rename(columns={'Student_ID': 'student_id'})
df_to_save.to_parquet(output_file_path, index=False)

print(f"Feature DataFrame successfully re-saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame successfully re-saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Step 2: Ensure feature_store.py is correctly defined.
# This cell writes the feature store definition to the correct location.
import os

feature_store_content = '''
from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)
'''

feature_store_file_path = os.path.join(FEAST_PROJECT_NAME, 'feature_store.py')
with open(feature_store_file_path, 'w') as f:
    f.write(feature_store_content)

print(f"Feast feature definitions saved to {feature_store_file_path}")
print(f"\nVerifying content of {feature_store_file_path}:\n{feature_store_content}")

Feast feature definitions saved to /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py

Verifying content of /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py:

from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    source=employability_data_source,
)



In [ ]:
# Step 3: Re-apply Feast definitions.
# This step registers the updated entity and feature view with the Feast registry.
import os

if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run the Feast initialization cell first.")
else:
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")
    %cd {FEAST_PROJECT_NAME}
    print("\nRunning feast apply...")
    !feast apply
    %cd /content # Change back to original directory
    print("\nFeast apply command executed.")

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Running feast apply...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/sty

In [ ]:
# Step 4: Materialize features into the online store.
# Correcting the typo in materialization_end_time variable.
from datetime import datetime, timedelta
from feast import FeatureStore # Re-initialize FeatureStore to ensure context

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

# Find the minimum and maximum event_timestamp from the original DataFrame
min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

# Define the materialization window
materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

store.materialize_incremental(
    end_date=materialization_end_time
)

print("\nFeatures successfully materialized into the online store.")

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Step 5: Retrieve historical features.
# Correcting the FeatureRequest import path and usage.
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

# Initialize the feature store client
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

# Prepare an entity_df for historical feature retrieval
# Ensure 'Student_ID' is still in the DataFrame for this step, though 'student_id' is used internally by Feast
entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
# Re-save the pre-processed DataFrame to a Parquet file with the corrected 'student_id' column (original cell 38694e83)
# This is crucial for materialization to find the correct entity key.
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
df_to_save = df.rename(columns={'Student_ID': 'student_id'})
df_to_save.to_parquet(output_file_path, index=False)
print(f"Feature DataFrame saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Re-apply Feast definitions (original cell 62c8e296)
# This ensures any implicit schema changes due to data update are registered.
import os
import inspect
from feast import FeatureView # Import FeatureView for introspection

if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run cell bf0f1d9a first.")
else:
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")
    print(f"Changing directory to: {FEAST_PROJECT_NAME}")
    %cd {FEAST_PROJECT_NAME}

    print("\nFeast CLI Version:")
    !feast version

    print("\nFeatureView __init__ signature:")
    try:
        signature = inspect.signature(FeatureView.__init__)
        print(signature)
    except Exception as e:
        print(f"Could not get FeatureView signature: {e}")

    print("\nRunning feast apply...")
    !feast apply

    print("\nChanging back to /content directory...")
    %cd /content

    print("\nFeast apply command executed.")

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
Changing directory to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Feast CLI Version:
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enab

In [ ]:
# Re-run materialization into the online store (original cell ccccaa74 or 75111b1f)
# Now with the corrected 'timedelta' import and 'student_id' in Parquet.
from datetime import datetime, timedelta
from feast import FeatureStore # Need to re-initialize store as context might be lost

store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

store.materialize_incremental(
    end_date=materialization_end_time
)

print("\nFeatures successfully materialized into the online store.")

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Re-run historical feature retrieval (original cell d3c43f96 or 6e18159f)
# Now with the corrected 'FeatureRequest' import path.
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
# Re-save the Parquet file with the corrected 'student_id' column (original cell 38694e83)
# This is crucial for materialization to find the correct entity key.
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
df_to_save = df.rename(columns={'Student_ID': 'student_id'})
df_to_save.to_parquet(output_file_path, index=False)
print(f"Feature DataFrame saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
# Re-apply Feast definitions (original cell 62c8e296)
# This ensures any implicit schema changes due to data update are registered.
import os
import inspect
from feast import FeatureView # Import FeatureView for introspection

if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run cell bf0f1d9a first.")
else:
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")
    print(f"Changing directory to: {FEAST_PROJECT_NAME}")
    %cd {FEAST_PROJECT_NAME}

    print("\nFeast CLI Version:")
    !feast version

    print("\nFeatureView __init__ signature:")
    try:
        signature = inspect.signature(FeatureView.__init__)
        print(signature)
    except Exception as e:
        print(f"Could not get FeatureView signature: {e}")

    print("\nRunning feast apply...")
    !feast apply

    print("\nChanging back to /content directory...")
    %cd /content

    print("\nFeast apply command executed.")

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
Changing directory to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Feast CLI Version:
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enab

In [ ]:
# Re-run materialization into the online store (original cell ccccaa74)
# Now with the corrected 'timedelta' import and 'student_id' in Parquet.
from datetime import datetime, timedelta

min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

store.materialize_incremental(
    end_date=materialization_end_time
)

print("\nFeatures successfully materialized into the online store.")

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Re-run historical feature retrieval (original cell d3c43f96)
# Now with the corrected 'FeatureRequest' import path.
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
# Re-save the pre-processed DataFrame to a Parquet file within the Feast project directory
# Use the globally defined FEAST_PROJECT_NAME, which now holds the correct path
output_file_path = f'{FEAST_PROJECT_NAME}/data/employability_features.parquet'

# Ensure the directory exists
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

# Rename 'Student_ID' column to 'student_id' to match the Feast entity definition
df_to_save = df.rename(columns={'Student_ID': 'student_id'})

df_to_save.to_parquet(output_file_path, index=False)

print(f"Feature DataFrame saved to {output_file_path} with 'student_id' as entity key.")

Feature DataFrame saved to /content/feast_container_root/wondrous_shiner/feature_repo/data/employability_features.parquet with 'student_id' as entity key.


In [ ]:
import os
import inspect
from feast import FeatureView # Import FeatureView for introspection

# Ensure the global FEAST_PROJECT_NAME is accessible
# (It should have been set by the previous cell bf0f1d9a)
if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run cell bf0f1d9a first.")
else:
    # Set FEAST_REPO_PATH environment variable to explicitly tell Feast where to find the repository
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")

    # Although FEAST_REPO_PATH is set, still good practice to cd into the directory
    # to ensure any relative paths in feature_store.py are resolved correctly.
    print(f"Changing directory to: {FEAST_PROJECT_NAME}")
    %cd {FEAST_PROJECT_NAME}

    # Diagnostic: Print Feast CLI version
    print("\nFeast CLI Version:")
    !feast version

    # Diagnostic: Introspect FeatureView constructor arguments
    print("\nFeatureView __init__ signature:")
    try:
        signature = inspect.signature(FeatureView.__init__)
        print(signature)
    except Exception as e:
        print(f"Could not get FeatureView signature: {e}")

    # Apply the Feast definitions
    print("\nRunning feast apply...")
    !feast apply

    # Change back to the original /content directory
    print("\nChanging back to /content directory...")
    %cd /content

    print("\nFeast apply command executed.")
    # Unset the environment variable to clean up, if desired
    # del os.environ["FEAST_REPO_PATH"]

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
Changing directory to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Feast CLI Version:
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enab

In [ ]:
from datetime import datetime, timedelta

# Get the current time for materialization window
# The start time is set to a day before the earliest event_timestamp in the data (2023-01-01)
# The end time is now(). This ensures all historical data is materialized.

# Find the minimum and maximum event_timestamp from the original DataFrame
min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

# Define the materialization window
# We'll materialize from just before the first event to slightly after the last event in our dataset
# This ensures all data points are covered.
materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

# Materialize features into the online store
store.materialize_incremental(
    end_date=materialization_end_time # Corrected typo here
)

print("\nFeatures successfully materialized into the online store.")

# Optional: Verify the online store status (e.g., list entities, feature views)
# This does not directly show data but confirms the store is aware of materialized features.
# print("\nOnline store status (entities and feature views):")
# !feast entities
# !feast feature-views

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2023-01-01 00:50:59+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

# Initialize the feature store client
# FEAST_REPO_PATH is globally defined from the Feast initialization step
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

# Prepare an entity_df for historical feature retrieval
# We'll use a subset of our original DataFrame for this example
# Ensure 'student_id' is the entity column and 'event_timestamp' is present
entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

# Retrieve historical features
training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    # Changed 'feature_views' to 'feature_requests' and used FeatureRequest object
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
# Re-save the Parquet file with the corrected 'student_id' column
%rerun 38694e83

No lines in history match specification


In [ ]:
# Re-apply Feast definitions (this will update the registry with the correct entity mapping if any implicit changes occur)
%rerun 62c8e296

No lines in history match specification


In [ ]:
# Re-run materialization into the online store after fixing the 'timedelta' import and 'student_id' column issue
%rerun ccccaa74

No lines in history match specification


In [ ]:
# Re-run historical feature retrieval after fixing the 'FeatureRequest' import
%rerun d3c43f96

No lines in history match specification


In [ ]:
# Re-save the Parquet file with the corrected 'student_id' column
%rerun 38694e83

No lines in history match specification


In [ ]:
# Re-apply Feast definitions (this will update the registry with the correct entity mapping if any implicit changes occur)
%rerun 62c8e296

No lines in history match specification


In [ ]:
# Re-run materialization into the online store after fixing the 'timedelta' import and 'student_id' column issue
%rerun ccccaa74

No lines in history match specification


In [ ]:
# Re-run historical feature retrieval after fixing the 'FeatureRequest' import
%rerun d3c43f96

No lines in history match specification


In [ ]:
import os

# Define the content for feature_store.py
feature_store_content = '''
from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    # Removed 'features' argument as it's not supported in Feast 0.65.0 FeatureView constructor.
    # Feast will infer features from the batch_source's schema.
    source=employability_data_source, # Corrected to use 'source' instead of 'batch_source'
)

'''

# Write the content to feature_store.py inside the Feast project directory
# Use the globally defined FEAST_PROJECT_NAME, which now holds the correct path
feature_store_file_path = os.path.join(FEAST_PROJECT_NAME, 'feature_store.py')
with open(feature_store_file_path, 'w') as f:
    f.write(feature_store_content)

print(f"Feast feature definitions saved to {feature_store_file_path}")

# Verify the content of the written file
print(f"\nVerifying content of {feature_store_file_path}:")
with open(feature_store_file_path, 'r') as f:
    print(f.read())


Feast feature definitions saved to /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py

Verifying content of /content/feast_container_root/wondrous_shiner/feature_repo/feature_store.py:

from feast import Entity, FeatureView, FileSource, ValueType
from datetime import timedelta

# Define an entity for students
student = Entity(
    name="student_id",
    value_type=ValueType.STRING,
    description="Unique identifier for a student",
)

# Define a data source from the Parquet file
employability_data_source = FileSource(
    path="data/employability_features.parquet",  # Relative path within the Feast project directory
    timestamp_field="event_timestamp",
)

# Define a FeatureView for employability features
employability_fv = FeatureView(
    name="employability_features",
    entities=[student], # Use the defined entity object
    ttl=timedelta(days=365),
    # Removed 'features' argument as it's not supported in Feast 0.65.0 FeatureView constructor.
    # Feas

In [ ]:
import os
import inspect
from feast import FeatureView # Import FeatureView for introspection

# Ensure the global FEAST_PROJECT_NAME is accessible
# (It should have been set by the previous cell bf0f1d9a)
if 'FEAST_PROJECT_NAME' not in globals():
    print("Error: FEAST_PROJECT_NAME is not defined. Please run cell bf0f1d9a first.")
else:
    # Set FEAST_REPO_PATH environment variable to explicitly tell Feast where to find the repository
    os.environ["FEAST_REPO_PATH"] = FEAST_PROJECT_NAME
    print(f"FEAST_REPO_PATH set to: {os.environ['FEAST_REPO_PATH']}")

    # Although FEAST_REPO_PATH is set, still good practice to cd into the directory
    # to ensure any relative paths in feature_store.py are resolved correctly.
    print(f"Changing directory to: {FEAST_PROJECT_NAME}")
    %cd {FEAST_PROJECT_NAME}

    # Diagnostic: Print Feast CLI version
    print("\nFeast CLI Version:")
    !feast version

    # Diagnostic: Introspect FeatureView constructor arguments
    print("\nFeatureView __init__ signature:")
    try:
        signature = inspect.signature(FeatureView.__init__)
        print(signature)
    except Exception as e:
        print(f"Could not get FeatureView signature: {e}")

    # Apply the Feast definitions
    print("\nRunning feast apply...")
    !feast apply

    # Change back to the original /content directory
    print("\nChanging back to /content directory...")
    %cd /content

    print("\nFeast apply command executed.")
    # Unset the environment variable to clean up, if desired
    # del os.environ["FEAST_REPO_PATH"]

FEAST_REPO_PATH set to: /content/feast_container_root/wondrous_shiner/feature_repo
Changing directory to: /content/feast_container_root/wondrous_shiner/feature_repo
/content/feast_container_root/wondrous_shiner/feature_repo

Feast CLI Version:
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enab

In [8]:
from feast import FeatureStore
import pandas as pd

store = FeatureStore(repo_path=FEAST_PROJECT_NAME)
entity_df = df[['Student_ID', 'event_timestamp']].head(5).rename(columns={'Student_ID': 'student_id'})

# Retrieve historical features for training
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "employability_features:CGPA",
        "employability_features:total_technical_skills_score",
        "employability_features:total_soft_skills_score",
        "employability_features:total_extracurricular_score",
        "employability_features:overall_readiness_score",
        "employability_features:Industry_Ready_Encoded"
    ]
).to_df()

print("Historical features retrieved successfully.")
print(training_df.head())

NameError: name 'FEAST_PROJECT_NAME' is not defined

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from feast import FeatureStore, FeatureRequest # Corrected import for FeatureRequest

# Initialize the feature store client
# FEAST_REPO_PATH is globally defined from the Feast initialization step
store = FeatureStore(repo_path=FEAST_PROJECT_NAME)

print(f"Feast store initialized using repository: {store.repo_path}")

# Prepare an entity_df for historical feature retrieval
# We'll use a subset of our original DataFrame for this example
# Ensure 'student_id' is the entity column and 'event_timestamp' is present
entity_df_for_historical = df[['Student_ID', 'event_timestamp']].head(5)

print("\nEntity DataFrame for historical retrieval (first 5 rows):")
print(entity_df_for_historical)

# Retrieve historical features
training_df = store.get_historical_features(
    entity_df=entity_df_for_historical,
    # Changed 'feature_views' to 'feature_requests' and used FeatureRequest object
    feature_requests=[FeatureRequest(feature_view_name="employability_features")]
).to_df()

print("\nRetrieved Historical Features (first 5 rows):")
print(training_df.head())
print("\nSchema of Retrieved Historical Features:")
print(training_df.info())

ImportError: cannot import name 'FeatureRequest' from 'feast' (/usr/local/lib/python3.12/dist-packages/feast/__init__.py)

In [ ]:
from datetime import datetime, timedelta

# Get the current time for materialization window
# The start time is set to a day before the earliest event_timestamp in the data (2023-01-01)
# The end time is now(). This ensures all historical data is materialized.

# Find the minimum and maximum event_timestamp from the original DataFrame
min_event_timestamp = df['event_timestamp'].min()
max_event_timestamp = df['event_timestamp'].max()

# Define the materialization window
# We'll materialize from just before the first event to slightly after the last event in our dataset
# This ensures all data points are covered.
materialization_start_time = min_event_timestamp - timedelta(days=1)
materialization_end_time = max_event_timestamp + timedelta(minutes=1)

print(f"Materializing features from {materialization_start_time} to {materialization_end_time}")

# Materialize features into the online store
store.materialize_incremental(
    end_date=materialization_end_time # Corrected typo here
)

print("\nFeatures successfully materialized into the online store.")

# Optional: Verify the online store status (e.g., list entities, feature views)
# This does not directly show data but confirms the store is aware of materialized features.
# print("\nOnline store status (entities and feature views):")
# !feast entities
# !feast feature-views

Materializing features from 2022-12-31 00:00:00 to 2023-01-01 00:50:59
Materializing 1 feature views to 2023-01-01 00:50:59+00:00 into the sqlite online store.

employability_features from 2025-08-17 06:22:07+00:00 to 2023-01-01 00:50:59+00:00:

Features successfully materialized into the online store.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Re-run materialization after fixing the timedelta import
%rerun ccccaa74

No lines in history match specification


In [ ]:
# Re-run historical feature retrieval after fixing the FeatureRequest issue
%rerun d3c43f96

No lines in history match specification


In [ ]:
# Re-run materialization after fixing the timedelta import
%rerun ccccaa74

No lines in history match specification


In [ ]:
# Re-run historical feature retrieval after fixing the FeatureRequest issue
%rerun d3c43f96

No lines in history match specification


In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.3 MB/s eta 0:00:00
